# Tarea 1 - Pipeline de Datos

Implementación de un pipeline con arquitectura Medallion (Bronze, Silver, Gold). 

In [0]:
# Carga de datos 
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", "|") \
    .load("/Volumes/clases_spark/intro/recursos/sessions_part1.csv")

display(df)


In [0]:
from pyspark.sql.functions import col, to_timestamp, date_trunc

df_bronze = df.withColumn(
    "event_time",
    date_trunc("hour", to_timestamp(col("timestamp")))
)

df_bronze.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("event_time") \
    .saveAsTable("tarea1_bronze.sessions")

In [0]:
%sql
SELECT * FROM tarea1_bronze.sessions LIMIT 10;

In [0]:
df_bronze = spark.table("tarea1_bronze.sessions")
display(df_bronze)

In [0]:
from pyspark.sql.functions import schema_of_json, col

# tomar un ejemplo
sample_json = df_bronze.select("data").first()[0]

schema = schema_of_json(sample_json)

In [0]:
from pyspark.sql.functions import from_json

df_silver = df_bronze.withColumn(
    "json",
    from_json(col("data"), schema)
).select("json.*", "event_time")

print(df_silver.printSchema())


In [0]:
df_silver = df_silver.drop("packetPos")

In [0]:
display(df_silver)

In [0]:
from pyspark.sql.functions import when

df_silver = df_silver.withColumn(
    "packetlen_sum",
    expr("aggregate(packetLen, CAST(0 AS BIGINT), (acc, x) -> acc + x)")
).withColumn(
    "packetlen_mean",
    when(size(col("packetLen")) > 0, col("packetlen_sum") / size(col("packetLen")))
).withColumn(
    "packetlen_min",
    array_min(col("packetLen"))
).withColumn(
    "packetlen_max",
    array_max(col("packetLen"))
)

In [0]:
df_silver = df_silver.drop("packetLen")

In [0]:
from pyspark.sql.functions import from_unixtime

df_silver = df_silver.withColumn(
    "firstPacket",
    from_unixtime(col("firstPacket"))
).withColumn(
    "lastPacket",
    from_unixtime(col("lastPacket"))
).withColumn(
    "timestamp",
    from_unixtime(col("timestamp"))
)

In [0]:
import re

def to_snake(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

df_silver = df_silver.toDF(*[to_snake(c) for c in df_silver.columns])

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("tarea1_silver.sessions")

In [0]:
%sql
SELECT * FROM tarea1_silver.sessions LIMIT 10;

## Diccionario de Datos (Capa Silver)

A continuación se presenta una descripción del posible significado de los campos contenidos en el dataset de sesiones de red:

- **communityId**: Identificador único de la sesión de red, utilizado para correlacionar tráfico entre origen y destino.
- **srcIp / dstIp**: Dirección IP de origen y destino de la comunicación.
- **srcPort / dstPort**: Puertos de origen y destino utilizados en la conexión.
- **srcASN / dstASN**: Sistema Autónomo (ASN) asociado a las direcciones IP de origen y destino.
- **srcGEO / dstGEO**: Ubicación geográfica (país o región) asociada a las IPs.
- **srcMac / dstMac**: Direcciones MAC involucradas en la comunicación.
- **srcMacCnt / dstMacCnt**: Número de direcciones MAC observadas en origen y destino.
- **srcBytes / dstBytes**: Cantidad total de bytes enviados desde el origen y recibidos por el destino.
- **srcDataBytes / dstDataBytes**: Bytes útiles de datos (excluyendo overhead de protocolo).
- **totBytes**: Total de bytes transferidos en la sesión.
- **totDataBytes**: Total de bytes de datos útiles en la sesión.
- **srcPackets / dstPackets**: Número de paquetes enviados y recibidos.
- **totPackets**: Número total de paquetes en la sesión.
- **packetLen**: Lista con el tamaño de cada paquete en la sesión.
- **packetPos**: Posición o secuencia de los paquetes dentro de la sesión.
- **protocol**: Lista de protocolos utilizados (por ejemplo, TCP, HTTP).
- **protocolCnt**: Número de protocolos identificados en la sesión.
- **ipProtocol**: Código numérico del protocolo IP (por ejemplo, 6 para TCP, 17 para UDP).
- **tcpflags**: Estructura que indica las banderas TCP (SYN, ACK, FIN, etc.).
- **segmentCnt**: Número de segmentos en la sesión.
- **length**: Duración o longitud de la sesión.
- **firstPacket / lastPacket**: Timestamp del primer y último paquete de la sesión.
- **timestamp**: Marca de tiempo asociada al evento.
- **event_time**: Timestamp procesado en formato datetime para análisis temporal.
- **node**: Nodo o dispositivo donde se capturó la información de la red.
- **fileId**: Identificador(es) del archivo(s) asociado(s) a la sesión.